In [1]:
import os
os.environ["PYSPARK_PYTHON"] = 'python'
import sqlite3
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [2]:
DB_NAME = "BODSDatabase.db"

In [3]:
conn = sqlite3.connect(DB_NAME)
conn.execute("PRAGMA foreign_keys = ON")

In [4]:
conn.execute("""
CREATE TABLE IF NOT EXISTS dim_line (
    line_code TEXT PRIMARY KEY
)
""")

In [5]:
conn.execute("""
CREATE TABLE IF NOT EXISTS fact_timetable (
    row_id INTEGER PRIMARY KEY AUTOINCREMENT,
    line_code TEXT REFERENCES dim_line(line_code),
    ServiceCode TEXT, LineName TEXT, JourneyCode TEXT,
    StopPointRef TEXT, StopName TEXT, RunTime TEXT, Source TEXT
)
""")

In [6]:
conn.execute("""
CREATE TABLE IF NOT EXISTS fact_fare (
    row_id INTEGER PRIMARY KEY AUTOINCREMENT,
    line_code TEXT REFERENCES dim_line(line_code),
    LineID TEXT, Amount REAL, Currency TEXT, Source TEXT
)
""")

In [7]:
conn.execute("""
CREATE TABLE IF NOT EXISTS fact_location (
    row_id INTEGER PRIMARY KEY AUTOINCREMENT,
    line_code TEXT REFERENCES dim_line(line_code),
    LineRef TEXT, RecordedAtTime TEXT, DatedVehicleJourneyRef TEXT,
    OperatorRef TEXT, OriginRef TEXT, OriginName TEXT,
    DestinationRef TEXT, DestinationName TEXT,
    OriginAimedDepartureTime TEXT, Longitude REAL, Latitude REAL, VehicleRef TEXT
)
""")

In [8]:
conn.execute("""
CREATE TABLE IF NOT EXISTS fact_disruption (
    row_id INTEGER PRIMARY KEY AUTOINCREMENT,
    line_code TEXT REFERENCES dim_line(line_code),
    SituationNumber TEXT, AffectedStopRef TEXT, AffectedStopName TEXT,
    Severity TEXT, ReasonCategory TEXT, ReasonType TEXT, Planned INTEGER,
    StartTime TEXT, EndTime TEXT, DurationHours REAL, IsSynthetic INTEGER
)
""")

In [9]:
conn.commit()
print(f"Schema created in {DB_NAME}")

Schema created in BODSDatabase.db


In [10]:
tt = pd.read_csv('E:\school(softwarica)\Sem-4\BigDataProject\Data\FinalCSV\clean_timetabledata.csv')
tt['line_code'] = tt['LineName']

In [11]:
fares = pd.read_csv('E:\school(softwarica)\Sem-4\BigDataProject\Data\FinalCSV\clean_fares.csv')
fares['line_code'] = fares['LineID'].str.split(':').str[-1]

In [12]:
loc = pd.read_csv('E:\school(softwarica)\Sem-4\BigDataProject\Data\FinalCSV\compiled_location_data.csv')
loc['line_code'] = loc['LineRef']

In [13]:
disr = pd.read_csv('E:\school(softwarica)\Sem-4\BigDataProject\Data\FinalCSV\clean_disruptiondata.csv')
disr['line_code'] = disr['LineRef']

In [14]:
all_lines = pd.concat([tt['line_code'], fares['line_code'], loc['line_code'], disr['line_code']]).dropna().unique()
pd.DataFrame({'line_code': sorted(all_lines)}).to_sql('dim_line', conn, if_exists='append', index = False)

44

In [15]:
tt[['line_code','ServiceCode','LineName','JourneyCode','StopPointRef','StopName','RunTime','Source']]\
    .to_sql('fact_timetable', conn, if_exists='append', index=False)
fares[['line_code','LineID','Amount','Currency','Source']]\
    .to_sql('fact_fare', conn, if_exists='append', index=False)
loc[['line_code','LineRef','RecordedAtTime','DatedVehicleJourneyRef','OperatorRef','OriginRef',
     'OriginName','DestinationRef','DestinationName','OriginAimedDepartureTime',
     'Longitude','Latitude','VehicleRef']]\
    .to_sql('fact_location', conn, if_exists='append', index=False)
disr[['line_code','SituationNumber','AffectedStopRef','AffectedStopName','Severity',
      'ReasonCategory','ReasonType','Planned','StartTime','EndTime','DurationHours','IsSynthetic']]\
    .to_sql('fact_disruption', conn, if_exists='append', index=False)
conn.commit()

In [16]:
for table in ['dim_line', 'fact_timetable', 'fact_fare', 'fact_location', 'fact_disruption']:
    n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}: {n} rows")

dim_line: 44 rows
fact_timetable: 176615 rows
fact_fare: 63428 rows
fact_location: 742 rows
fact_disruption: 170 rows


In [17]:
spark = (
    SparkSession.builder.appName("BusDisruptionAnalytics")
    .master("local[*]").getOrCreate()
)

In [18]:
timetable = spark.read.csv("E:\school(softwarica)\Sem-4\BigDataProject\Data\FinalCSV\clean_timetabledata.csv", header = True, inferSchema = True)
fares_sp = spark.read.csv("E:\school(softwarica)\Sem-4\BigDataProject\Data\FinalCSV\clean_fares.csv", header = True, inferSchema = True)
location_sp = spark.read.csv("E:\school(softwarica)\Sem-4\BigDataProject\Data\FinalCSV\compiled_location_data.csv", header = True, inferSchema = True)
disruption_sp = spark.read.csv("E:\school(softwarica)\Sem-4\BigDataProject\Data\FinalCSV\clean_disruptiondata.csv", header = True, inferSchema = True)

In [19]:
timetable = timetable.withColumn("line_code", col("LineName"))
fares_sp = fares_sp.withColumn("line_code", element_at(split(col("LineID"), ":"), -1))
location_sp = location_sp.withColumn("line_code", col("LineRef"))
disruption_sp = disruption_sp.withColumn("line_code", col("LineRef"))

In [20]:
timetable.createOrReplaceTempView("timetable")
fares_sp.createOrReplaceTempView("fares")
location_sp.createOrReplaceTempView("location")
disruption_sp.createOrReplaceTempView("disruption")

In [21]:
query4 = """
WITH tt_agg AS (
    SELECT line_code, COUNT(DISTINCT JourneyCode) AS n_scheduled_trips
    FROM timetable
    GROUP BY line_code
),
loc_agg AS (
    SELECT line_code, COUNT(DISTINCT DatedVehicleJourneyRef) AS n_avl_observations
    FROM location
    GROUP BY line_code
),
disr_agg AS (
    SELECT line_code, COUNT(DISTINCT SituationNumber) AS n_disruptions
    FROM disruption
    GROUP BY line_code
),
fares_agg AS (
    SELECT line_code, ROUND(AVG(Amount), 2) AS avg_fare
    FROM fares
    GROUP BY line_code
)

SELECT 
    t.line_code,
    COALESCE(t.n_scheduled_trips, 0) AS n_scheduled_trips,
    COALESCE(l.n_avl_observations, 0) AS n_avl_observations,
    COALESCE(d.n_disruptions, 0) AS n_disruptions,
    COALESCE(f.avg_fare, 0.0) AS avg_fare
FROM tt_agg t
LEFT JOIN loc_agg l ON t.line_code = l.line_code
LEFT JOIN disr_agg d ON t.line_code = d.line_code
LEFT JOIN fares_agg f ON t.line_code = f.line_code
ORDER BY n_disruptions DESC
"""

In [22]:
result4 = spark.sql(query4)
result4.show(50)

+---------+-----------------+------------------+-------------+--------+
|line_code|n_scheduled_trips|n_avl_observations|n_disruptions|avg_fare|
+---------+-----------------+------------------+-------------+--------+
|      200|               50|                 4|            7|    4.35|
|       2A|              355|                15|            7|    1.64|
|       B3|              124|                 1|            7|    1.82|
|       B9|              272|                 9|            7|    1.86|
|       29|               67|                 4|            6|    3.06|
|       S7|              162|                27|            6|    3.49|
|       S4|               80|                14|            6|     4.0|
|      233|               65|                 2|            6|    3.16|
|      B10|               20|                 2|            6|    1.65|
|     TUBE|              559|                52|            6|   14.43|
|       N1|               65|                 2|            6|  

In [23]:
result4_pandas = result4.toPandas()
result4_pandas.to_sql('line_summary', conn, if_exists='replace', index=False)
conn.commit()
conn.close()